# Experiments 02–03: MIA-Ready MLP and Baseline MIA Audit

This notebook performs two connected tasks:

1. Build the locked NSL-KDD research split and train the non-private target MLP.
2. Audit the target MLP with shadow-calibrated membership-inference attacks.

## Fixed protocol

- `KDDTrain+`
  - 70% target-model training
  - 10% target validation and primary MIA non-members
  - 20% shadow-model pool
- `KDDTest+`
  - IDS utility evaluation only
- Target model
  - MLP
- MIA calibration
  - Entire unseen shadow model, not random rows from pooled shadows
- Threat models
  - Score-only black-box
  - Label-aware audit


## 1. Environment and experiment configuration

In [ ]:
# Imports
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.max_columns", 100)

In [ ]:
# Experiment settings
SEED = 42

# Use "smoke" first. Change to "final", restart the runtime, and run all cells
# after the smoke test completes successfully.
RUN_MODE = "smoke"

SHADOW_SEEDS = {
    "smoke": [101, 202],
    "final": [101, 202, 303, 404, 505],
}[RUN_MODE]

BOOTSTRAP_N = {
    "smoke": 500,
    "final": 1000,
}[RUN_MODE]

random.seed(SEED)
np.random.seed(SEED)

print({
    "run_mode": RUN_MODE,
    "shadow_models": len(SHADOW_SEEDS),
    "bootstrap_samples": BOOTSTRAP_N,
})

In [ ]:
# Project paths
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

DEFAULT_PROJECT_DIR = Path("/content/drive/MyDrive/ML-DP-NID")
PROJECT_DIR = Path(os.environ.get("ML_DP_NID_DIR", DEFAULT_PROJECT_DIR))

if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

TRAIN_FILE = PROJECT_DIR / "KDDTrain+.txt"
TEST_FILE = PROJECT_DIR / "KDDTest+.txt"

RESULTS_DIR = PROJECT_DIR / "results" / "baseline_mia_combined"
FIGURES_DIR = RESULTS_DIR / "figures"
SPLIT_DIR = PROJECT_DIR / "data" / "split_indices"
MODEL_DIR = PROJECT_DIR / "artifacts" / "models"
PREPROCESSOR_DIR = PROJECT_DIR / "artifacts" / "preprocessor"
MANIFEST_DIR = PROJECT_DIR / "artifacts" / "manifests"

for directory in [
    RESULTS_DIR,
    FIGURES_DIR,
    SPLIT_DIR,
    MODEL_DIR,
    PREPROCESSOR_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

assert TRAIN_FILE.exists(), f"Missing dataset: {TRAIN_FILE}"
assert TEST_FILE.exists(), f"Missing dataset: {TEST_FILE}"

print("Project directory:", PROJECT_DIR)

## 2. NSL-KDD schema and label definitions

In [ ]:
# Dataset columns
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty",
]

FEATURES = COLUMNS[:41]
CATEGORICAL_FEATURES = ["protocol_type", "service", "flag"]
NUMERIC_FEATURES = [
    column for column in FEATURES
    if column not in CATEGORICAL_FEATURES
]

In [ ]:
# Coarse attack-family mapping
DOS_ATTACKS = {
    "back", "land", "neptune", "pod", "smurf", "teardrop",
    "apache2", "udpstorm", "processtable", "mailbomb",
}

PROBE_ATTACKS = {
    "satan", "ipsweep", "nmap", "portsweep", "mscan", "saint",
}

R2L_ATTACKS = {
    "guess_passwd", "ftp_write", "imap", "phf", "multihop",
    "warezmaster", "warezclient", "spy", "xlock", "xsnoop",
    "snmpguess", "snmpgetattack", "httptunnel", "sendmail", "named",
}

U2R_ATTACKS = {
    "buffer_overflow", "loadmodule", "rootkit", "perl",
    "sqlattack", "xterm", "ps",
}

def map_attack_family(label: str) -> str:
    label = str(label).strip().lower().rstrip(".")

    if label == "normal":
        return "Normal"
    if label in DOS_ATTACKS:
        return "DoS"
    if label in PROBE_ATTACKS:
        return "Probe"
    if label in R2L_ATTACKS:
        return "R2L"
    if label in U2R_ATTACKS:
        return "U2R"

    return "OtherAttack"

def pool_rare_families(attack_family: str) -> str:
    if attack_family in {"Normal", "DoS", "Probe"}:
        return attack_family
    return "Rare"

In [ ]:
# Data-loading helpers
def calculate_sha256(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

def load_nsl_kdd(path: Path) -> pd.DataFrame:
    dataframe = pd.read_csv(path, names=COLUMNS)

    dataframe["row_id"] = np.arange(len(dataframe))
    dataframe["label_clean"] = (
        dataframe["label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.rstrip(".")
    )
    dataframe["binary_label"] = (
        dataframe["label_clean"] != "normal"
    ).astype(int)
    dataframe["attack_family"] = (
        dataframe["label_clean"].map(map_attack_family)
    )
    dataframe["family_group"] = (
        dataframe["attack_family"].map(pool_rare_families)
    )

    for column in NUMERIC_FEATURES:
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="raise",
        )

    return dataframe

## 3. Load data and create the locked research split

In [ ]:
# Load the official NSL-KDD files
train_df = load_nsl_kdd(TRAIN_FILE)
test_df = load_nsl_kdd(TEST_FILE)

print("KDDTrain+ rows:", len(train_df))
print("KDDTest+ rows:", len(test_df))

In [ ]:
# Step 1: create the 70% target-training partition
full_strata = (
    train_df["binary_label"].astype(str)
    + "__"
    + train_df["attack_family"]
)

target_train_indices, remaining_indices = train_test_split(
    np.arange(len(train_df)),
    test_size=0.30,
    random_state=SEED,
    stratify=full_strata,
)

remaining_df = train_df.iloc[remaining_indices]
remaining_strata = (
    remaining_df["binary_label"].astype(str)
    + "__"
    + remaining_df["attack_family"]
)


In [ ]:
# Step 2: divide the remaining 30% into 10% validation and 20% shadow pool
target_validation_indices, shadow_pool_indices = train_test_split(
    remaining_indices,
    test_size=2 / 3,
    random_state=SEED,
    stratify=remaining_strata,
)

target_train = (
    train_df.iloc[target_train_indices]
    .copy()
    .reset_index(drop=True)
)

target_validation = (
    train_df.iloc[target_validation_indices]
    .copy()
    .reset_index(drop=True)
)

shadow_pool = (
    train_df.iloc[shadow_pool_indices]
    .copy()
    .reset_index(drop=True)
)


In [ ]:
# Save split indices and verify the locked partition
np.save(
    SPLIT_DIR / "target_train_indices.npy",
    target_train_indices,
)
np.save(
    SPLIT_DIR / "target_validation_indices.npy",
    target_validation_indices,
)
np.save(
    SPLIT_DIR / "shadow_pool_indices.npy",
    shadow_pool_indices,
)

assert len(target_train) == 88181
assert len(target_validation) == 12597
assert len(shadow_pool) == 25195

all_indices = np.concatenate([
    target_train_indices,
    target_validation_indices,
    shadow_pool_indices,
])

assert len(np.unique(all_indices)) == len(train_df)
assert set(all_indices) == set(range(len(train_df)))

split_summary = pd.DataFrame([
    {
        "partition": "target_train",
        "rows": len(target_train),
        "normal": int((target_train["binary_label"] == 0).sum()),
        "attack": int((target_train["binary_label"] == 1).sum()),
    },
    {
        "partition": "target_validation",
        "rows": len(target_validation),
        "normal": int((target_validation["binary_label"] == 0).sum()),
        "attack": int((target_validation["binary_label"] == 1).sum()),
    },
    {
        "partition": "shadow_pool",
        "rows": len(shadow_pool),
        "normal": int((shadow_pool["binary_label"] == 0).sum()),
        "attack": int((shadow_pool["binary_label"] == 1).sum()),
    },
    {
        "partition": "KDDTest+",
        "rows": len(test_df),
        "normal": int((test_df["binary_label"] == 0).sum()),
        "attack": int((test_df["binary_label"] == 1).sum()),
    },
])

display(split_summary)


## 4. Build and train the MIA-ready target MLP

In [ ]:
# Model and preprocessing factories
def create_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

def create_preprocessor() -> ColumnTransformer:
    return ColumnTransformer([
        (
            "numeric",
            MinMaxScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            create_one_hot_encoder(),
            CATEGORICAL_FEATURES,
        ),
    ])

def create_mlp(seed: int) -> MLPClassifier:
    return MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=256,
        learning_rate_init=1e-3,
        max_iter=30,
        random_state=seed,
    )

def get_attack_probability(model, transformed_features) -> np.ndarray:
    classes = np.asarray(model.classes_)
    positive_class_index = int(
        np.where(classes == 1)[0][0]
    )
    return model.predict_proba(
        transformed_features
    )[:, positive_class_index]

In [ ]:
# Fit preprocessing on target_train only
target_preprocessor = create_preprocessor()

X_target_train = target_preprocessor.fit_transform(
    target_train[FEATURES]
)
X_target_validation = target_preprocessor.transform(
    target_validation[FEATURES]
)
X_test = target_preprocessor.transform(
    test_df[FEATURES]
)

print("Transformed feature count:", X_target_train.shape[1])

In [ ]:
# Train the non-private target MLP
target_model = create_mlp(SEED)

target_model.fit(
    X_target_train,
    target_train["binary_label"],
)

validation_scores = get_attack_probability(
    target_model,
    X_target_validation,
)
test_scores = get_attack_probability(
    target_model,
    X_test,
)

target_training_diagnostics = pd.DataFrame([{
    "seed": SEED,
    "n_iter": int(target_model.n_iter_),
    "max_iter": int(target_model.max_iter),
    "reached_max_iter": bool(target_model.n_iter_ >= target_model.max_iter),
    "final_loss": float(target_model.loss_),
}])

target_training_diagnostics.to_csv(
    RESULTS_DIR / "target_training_diagnostics.csv",
    index=False,
)

display(target_training_diagnostics)


## 5. Select the IDS threshold and evaluate utility

In [ ]:
# Search for the validation threshold that maximizes F2
threshold_rows = []

for threshold in np.arange(0.01, 1.00, 0.01):
    validation_predictions = (
        validation_scores >= threshold
    ).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "f2": fbeta_score(
            target_validation["binary_label"],
            validation_predictions,
            beta=2,
            zero_division=0,
        ),
        "recall": recall_score(
            target_validation["binary_label"],
            validation_predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            target_validation["binary_label"],
            validation_predictions,
            zero_division=0,
        ),
    })

threshold_search = pd.DataFrame(threshold_rows)

best_threshold = float(
    threshold_search
    .sort_values(
        ["f2", "recall"],
        ascending=False,
    )
    .iloc[0]["threshold"]
)

print("Selected validation F2 threshold:", best_threshold)

In [ ]:
# IDS metric helper
def calculate_ids_metrics(
    y_true,
    scores,
    threshold: float,
    split_name: str,
    threshold_policy: str,
) -> dict:
    y_true = np.asarray(y_true, dtype=int)
    predictions = (
        np.asarray(scores) >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "split": split_name,
        "threshold_policy": threshold_policy,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "fnr": fn / (fn + tp) if fn + tp else np.nan,
        "fpr": fp / (fp + tn) if fp + tn else np.nan,
        "roc_auc": roc_auc_score(y_true, scores),
        "pr_auc": average_precision_score(y_true, scores),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [ ]:
# Evaluate default and validation-selected thresholds
ids_results = pd.DataFrame([
    calculate_ids_metrics(
        target_validation["binary_label"],
        validation_scores,
        threshold=0.5,
        split_name="target_validation",
        threshold_policy="default_0_5",
    ),
    calculate_ids_metrics(
        test_df["binary_label"],
        test_scores,
        threshold=0.5,
        split_name="KDDTest+",
        threshold_policy="default_0_5",
    ),
    calculate_ids_metrics(
        test_df["binary_label"],
        test_scores,
        threshold=best_threshold,
        split_name="KDDTest+",
        threshold_policy="validation_selected_F2",
    ),
])

display(ids_results)

In [ ]:
# Save the target model, preprocessor, and IDS outputs
target_score_rows = pd.concat([
    pd.DataFrame({
        "split": "target_validation",
        "row_id": target_validation["row_id"],
        "true_label": target_validation["binary_label"],
        "family_group": target_validation["family_group"],
        "prob_attack": validation_scores,
    }),
    pd.DataFrame({
        "split": "KDDTest+",
        "row_id": test_df["row_id"],
        "true_label": test_df["binary_label"],
        "family_group": test_df["family_group"],
        "prob_attack": test_scores,
    }),
], ignore_index=True)

joblib.dump(
    target_preprocessor,
    PREPROCESSOR_DIR / "target_mlp_preprocessor.joblib",
)
joblib.dump(
    target_model,
    MODEL_DIR / "target_mlp.joblib",
)

threshold_search.to_csv(
    RESULTS_DIR / "target_threshold_search.csv",
    index=False,
)
ids_results.to_csv(
    RESULTS_DIR / "target_ids_results.csv",
    index=False,
)
target_score_rows.to_csv(
    RESULTS_DIR / "target_per_sample_scores.csv",
    index=False,
)

## 6. Construct the target MIA evaluation set

In [ ]:
# Convert target-model outputs into MIA features
PROBABILITY_EPSILON = 1e-12

def extract_mia_features(
    dataframe: pd.DataFrame,
    model,
    preprocessor,
    membership: int,
    source: str,
    shadow_seed: int | None = None,
) -> pd.DataFrame:
    transformed_features = preprocessor.transform(
        dataframe[FEATURES]
    )
    prob_attack = get_attack_probability(
        model,
        transformed_features,
    )

    true_labels = dataframe[
        "binary_label"
    ].to_numpy(dtype=int)

    true_class_probability = np.where(
        true_labels == 1,
        prob_attack,
        1 - prob_attack,
    )
    true_class_probability = np.clip(
        true_class_probability,
        PROBABILITY_EPSILON,
        1 - PROBABILITY_EPSILON,
    )

    return pd.DataFrame({
        "source": source,
        "shadow_seed": shadow_seed,
        "row_id": dataframe["row_id"].to_numpy(),
        "membership": membership,
        "true_label": true_labels,
        "binary_group": np.where(
            true_labels == 1,
            "Attack",
            "Normal",
        ),
        "family_group": dataframe[
            "family_group"
        ].to_numpy(),
        "prob_attack": prob_attack,
        "confidence": np.maximum(
            prob_attack,
            1 - prob_attack,
        ),
        "loss": -np.log(true_class_probability),
        "correctness": (
            (prob_attack >= 0.5).astype(int)
            == true_labels
        ).astype(int),
    })

In [ ]:
# Balance target members and non-members within each traffic group
def sample_balanced_target_records(
    member_candidates: pd.DataFrame,
    nonmember_candidates: pd.DataFrame,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    sampled_members = []
    sampled_nonmembers = []

    for group in ["Normal", "DoS", "Probe", "Rare"]:
        member_group = member_candidates[
            member_candidates["family_group"] == group
        ]
        nonmember_group = nonmember_candidates[
            nonmember_candidates["family_group"] == group
        ]

        sample_size = min(
            len(member_group),
            len(nonmember_group),
        )

        if sample_size == 0:
            continue

        sampled_members.append(
            member_group.sample(
                sample_size,
                random_state=seed,
            )
        )
        sampled_nonmembers.append(
            nonmember_group.sample(
                sample_size,
                random_state=seed + 1,
            )
        )

    members = pd.concat(
        sampled_members,
        ignore_index=True,
    )
    nonmembers = pd.concat(
        sampled_nonmembers,
        ignore_index=True,
    )

    return members, nonmembers

In [ ]:
# Create the fixed target MIA evaluation data
target_member_records, target_nonmember_records = (
    sample_balanced_target_records(
        target_train,
        target_validation,
        SEED,
    )
)

target_mia_data = pd.concat([
    extract_mia_features(
        target_member_records,
        target_model,
        target_preprocessor,
        membership=1,
        source="target_member",
    ),
    extract_mia_features(
        target_nonmember_records,
        target_model,
        target_preprocessor,
        membership=0,
        source="target_nonmember",
    ),
], ignore_index=True)

target_mia_data = (
    target_mia_data
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

target_mia_data.to_csv(
    RESULTS_DIR / "target_mia_evaluation.csv",
    index=False,
)

display(
    pd.crosstab(
        target_mia_data["family_group"],
        target_mia_data["membership"],
    )
)

## 7. Train shadow MLPs and generate attacker-development data

In [ ]:
# Split one shadow pool into members and non-members
def create_shadow_split(
    dataframe: pd.DataFrame,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    stratification_key = (
        dataframe["binary_label"].astype(str)
        + "__"
        + dataframe["attack_family"]
    )

    member_indices, nonmember_indices = train_test_split(
        np.arange(len(dataframe)),
        test_size=0.50,
        random_state=seed,
        stratify=stratification_key,
    )

    return (
        dataframe.iloc[member_indices].copy(),
        dataframe.iloc[nonmember_indices].copy(),
    )

In [ ]:
# Train all shadow MLPs
shadow_feature_frames = []
shadow_run_rows = []

for shadow_seed in SHADOW_SEEDS:
    shadow_members, shadow_nonmembers = create_shadow_split(
        shadow_pool,
        shadow_seed,
    )

    shadow_preprocessor = create_preprocessor()
    X_shadow_members = shadow_preprocessor.fit_transform(
        shadow_members[FEATURES]
    )

    shadow_model = create_mlp(shadow_seed)

    start_time = time.time()
    shadow_model.fit(
        X_shadow_members,
        shadow_members["binary_label"],
    )
    training_seconds = time.time() - start_time

    shadow_feature_frames.append(
        extract_mia_features(
            shadow_members,
            shadow_model,
            shadow_preprocessor,
            membership=1,
            source="shadow_member",
            shadow_seed=shadow_seed,
        )
    )
    shadow_feature_frames.append(
        extract_mia_features(
            shadow_nonmembers,
            shadow_model,
            shadow_preprocessor,
            membership=0,
            source="shadow_nonmember",
            shadow_seed=shadow_seed,
        )
    )

    shadow_run_rows.append({
        "shadow_seed": shadow_seed,
        "member_rows": len(shadow_members),
        "nonmember_rows": len(shadow_nonmembers),
        "training_seconds": training_seconds,
        "n_iter": int(shadow_model.n_iter_),
        "max_iter": int(shadow_model.max_iter),
        "reached_max_iter": bool(
            shadow_model.n_iter_ >= shadow_model.max_iter
        ),
        "final_loss": float(shadow_model.loss_),
    })

shadow_data = pd.concat(
    shadow_feature_frames,
    ignore_index=True,
)

shadow_data = (
    shadow_data
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

shadow_run_summary = pd.DataFrame(shadow_run_rows)
display(shadow_run_summary)


In [ ]:
# Use the final shadow model only for attacker calibration
calibration_shadow_seed = SHADOW_SEEDS[-1]

attack_train = shadow_data[
    shadow_data["shadow_seed"] != calibration_shadow_seed
].copy()

attack_calibration = shadow_data[
    shadow_data["shadow_seed"] == calibration_shadow_seed
].copy()

assert set(attack_train["shadow_seed"].unique()).isdisjoint(
    set(attack_calibration["shadow_seed"].unique())
)
assert attack_train["membership"].nunique() == 2
assert attack_calibration["membership"].nunique() == 2

shadow_data["attacker_split"] = np.where(
    shadow_data["shadow_seed"] == calibration_shadow_seed,
    "calibration",
    "train",
)

shadow_data.to_csv(
    RESULTS_DIR / "shadow_attack_data.csv",
    index=False,
)
shadow_run_summary.to_csv(
    RESULTS_DIR / "shadow_model_runs.csv",
    index=False,
)

print("Attacker training shadow seeds:", sorted(
    attack_train["shadow_seed"].unique().tolist()
))
print("Calibration shadow seed:", calibration_shadow_seed)
print("Attacker training rows:", len(attack_train))
print("Attacker calibration rows:", len(attack_calibration))


## 8. Calibrate the MIA attackers using shadow outputs only

In [ ]:
# General MIA utility functions
def calculate_mia_advantage(
    membership_labels,
    membership_scores,
) -> float:
    false_positive_rates, true_positive_rates, _ = roc_curve(
        membership_labels,
        membership_scores,
    )
    return float(
        np.max(
            true_positive_rates - false_positive_rates
        )
    )

def calculate_tpr_at_fpr(
    membership_labels,
    membership_scores,
    maximum_fpr: float,
) -> float:
    false_positive_rates, true_positive_rates, _ = roc_curve(
        membership_labels,
        membership_scores,
    )

    eligible = false_positive_rates <= maximum_fpr

    if not np.any(eligible):
        return 0.0

    return float(np.max(true_positive_rates[eligible]))

def find_balanced_accuracy_threshold(
    membership_labels,
    membership_scores,
) -> tuple[float, float]:
    candidate_thresholds = np.unique(
        np.quantile(
            membership_scores,
            np.linspace(0.001, 0.999, 400),
        )
    )

    best_threshold = float(candidate_thresholds[0])
    best_balanced_accuracy = -1.0

    for threshold in candidate_thresholds:
        predictions = membership_scores >= threshold
        balanced_accuracy = balanced_accuracy_score(
            membership_labels,
            predictions,
        )

        if balanced_accuracy > best_balanced_accuracy:
            best_threshold = float(threshold)
            best_balanced_accuracy = float(
                balanced_accuracy
            )

    return best_threshold, best_balanced_accuracy

In [ ]:
# Container for a calibrated MIA attack
@dataclass
class CalibratedAttack:
    threat_model: str
    attack_model: str
    feature_set: str
    operating_threshold: float
    shadow_calibration_auc: float
    score_function: Callable[
        [pd.DataFrame],
        np.ndarray,
    ]

In [ ]:
# Threshold-based attacker
def build_threshold_attack(
    threat_model: str,
    attack_model: str,
    feature_column: str,
    score_direction: float,
) -> CalibratedAttack:
    calibration_labels = attack_calibration[
        "membership"
    ].to_numpy()

    calibration_scores = (
        score_direction
        * attack_calibration[
            feature_column
        ].to_numpy()
    )

    operating_threshold, _ = (
        find_balanced_accuracy_threshold(
            calibration_labels,
            calibration_scores,
        )
    )

    return CalibratedAttack(
        threat_model=threat_model,
        attack_model=attack_model,
        feature_set=feature_column,
        operating_threshold=operating_threshold,
        shadow_calibration_auc=roc_auc_score(
            calibration_labels,
            calibration_scores,
        ),
        score_function=lambda dataframe: (
            score_direction
            * dataframe[feature_column].to_numpy()
        ),
    )

In [ ]:
# Learned attacker: Logistic Regression or Random Forest
def build_learned_attack(
    threat_model: str,
    attack_model: str,
    feature_columns: list[str],
) -> CalibratedAttack:
    if attack_model == "logistic_regression":
        model = Pipeline([
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=SEED,
                ),
            ),
        ])
    elif attack_model == "random_forest":
        model = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1,
        )
    else:
        raise ValueError(
            f"Unsupported attacker: {attack_model}"
        )

    model.fit(
        attack_train[feature_columns],
        attack_train["membership"],
    )

    calibration_scores = model.predict_proba(
        attack_calibration[feature_columns]
    )[:, 1]

    operating_threshold, _ = (
        find_balanced_accuracy_threshold(
            attack_calibration["membership"],
            calibration_scores,
        )
    )

    return CalibratedAttack(
        threat_model=threat_model,
        attack_model=attack_model,
        feature_set=" + ".join(feature_columns),
        operating_threshold=operating_threshold,
        shadow_calibration_auc=roc_auc_score(
            attack_calibration["membership"],
            calibration_scores,
        ),
        score_function=lambda dataframe: (
            model.predict_proba(
                dataframe[feature_columns]
            )[:, 1]
        ),
    )

In [ ]:
# Build the six planned MIA attackers
calibrated_attacks = [
    build_threshold_attack(
        threat_model="score_only_black_box",
        attack_model="confidence_threshold",
        feature_column="confidence",
        score_direction=1.0,
    ),
    build_learned_attack(
        threat_model="score_only_black_box",
        attack_model="logistic_regression",
        feature_columns=["prob_attack"],
    ),
    build_learned_attack(
        threat_model="score_only_black_box",
        attack_model="random_forest",
        feature_columns=["prob_attack"],
    ),
    build_threshold_attack(
        threat_model="label_aware_audit",
        attack_model="loss_threshold",
        feature_column="loss",
        score_direction=-1.0,
    ),
    build_learned_attack(
        threat_model="label_aware_audit",
        attack_model="logistic_regression",
        feature_columns=["loss", "correctness"],
    ),
    build_learned_attack(
        threat_model="label_aware_audit",
        attack_model="random_forest",
        feature_columns=["loss", "correctness"],
    ),
]

attack_calibration_summary = pd.DataFrame([
    {
        "threat_model": attack.threat_model,
        "attack_model": attack.attack_model,
        "feature_set": attack.feature_set,
        "operating_threshold": attack.operating_threshold,
        "shadow_calibration_auc": attack.shadow_calibration_auc,
    }
    for attack in calibrated_attacks
])

attack_calibration_summary.to_csv(
    RESULTS_DIR / "mia_attack_calibration.csv",
    index=False,
)

display(attack_calibration_summary)

## 9. Evaluate MIA performance on the fixed target data

In [ ]:
# Bootstrap confidence interval
def bootstrap_confidence_interval(
    membership_labels,
    membership_scores,
    metric_function,
    seed: int,
    bootstrap_samples: int = BOOTSTRAP_N,
) -> tuple[float, float]:
    random_generator = np.random.default_rng(seed)
    bootstrap_values = []

    for _ in range(bootstrap_samples):
        sample_indices = random_generator.integers(
            0,
            len(membership_labels),
            len(membership_labels),
        )

        sampled_labels = membership_labels[sample_indices]
        sampled_scores = membership_scores[sample_indices]

        if len(np.unique(sampled_labels)) < 2:
            continue

        bootstrap_values.append(
            metric_function(
                sampled_labels,
                sampled_scores,
            )
        )

    if not bootstrap_values:
        return np.nan, np.nan

    lower, upper = np.quantile(
        bootstrap_values,
        [0.025, 0.975],
    )

    return float(lower), float(upper)

In [ ]:
# Evaluate one calibrated attacker
def evaluate_calibrated_attack(
    attack: CalibratedAttack,
    evaluation_data: pd.DataFrame,
    subset_name: str,
    include_confidence_intervals: bool = True,
) -> dict:
    membership_labels = evaluation_data[
        "membership"
    ].to_numpy(dtype=int)

    membership_scores = attack.score_function(
        evaluation_data
    )

    membership_predictions = (
        membership_scores
        >= attack.operating_threshold
    ).astype(int)

    mia_auc = roc_auc_score(
        membership_labels,
        membership_scores,
    )
    mia_advantage = calculate_mia_advantage(
        membership_labels,
        membership_scores,
    )

    if include_confidence_intervals:
        auc_ci = bootstrap_confidence_interval(
            membership_labels,
            membership_scores,
            roc_auc_score,
            seed=SEED,
        )
        advantage_ci = bootstrap_confidence_interval(
            membership_labels,
            membership_scores,
            calculate_mia_advantage,
            seed=SEED + 1,
        )
    else:
        auc_ci = (np.nan, np.nan)
        advantage_ci = (np.nan, np.nan)

    return {
        "subset": subset_name,
        "threat_model": attack.threat_model,
        "attack_model": attack.attack_model,
        "feature_set": attack.feature_set,
        "n": len(evaluation_data),
        "members": int(membership_labels.sum()),
        "nonmembers": int(
            (1 - membership_labels).sum()
        ),
        "mia_auc": mia_auc,
        "mia_auc_ci_low": auc_ci[0],
        "mia_auc_ci_high": auc_ci[1],
        "mia_advantage": mia_advantage,
        "mia_advantage_ci_low": advantage_ci[0],
        "mia_advantage_ci_high": advantage_ci[1],
        "mia_balanced_accuracy": balanced_accuracy_score(
            membership_labels,
            membership_predictions,
        ),
        "mia_precision": precision_score(
            membership_labels,
            membership_predictions,
            zero_division=0,
        ),
        "mia_recall": recall_score(
            membership_labels,
            membership_predictions,
            zero_division=0,
        ),
        "tpr_at_1pct_fpr": calculate_tpr_at_fpr(
            membership_labels,
            membership_scores,
            maximum_fpr=0.01,
        ),
        "tpr_at_5pct_fpr": calculate_tpr_at_fpr(
            membership_labels,
            membership_scores,
            maximum_fpr=0.05,
        ),
        "operating_threshold": attack.operating_threshold,
        "shadow_calibration_auc": (
            attack.shadow_calibration_auc
        ),
    }

In [ ]:
# Overall target MIA results
mia_summary = pd.DataFrame([
    evaluate_calibrated_attack(
        attack,
        target_mia_data,
        subset_name="overall",
    )
    for attack in calibrated_attacks
])

mia_summary.to_csv(
    RESULTS_DIR / "mia_summary.csv",
    index=False,
)

display(
    mia_summary.sort_values(
        ["mia_auc", "mia_advantage"],
        ascending=False,
    )
)

In [ ]:
# Group diagnostics
# Select the best attacker for each threat model using shadow calibration only.
best_attack_by_threat_model = {}

for attack in calibrated_attacks:
    current_best = best_attack_by_threat_model.get(
        attack.threat_model
    )

    if (
        current_best is None
        or attack.shadow_calibration_auc
        > current_best.shadow_calibration_auc
    ):
        best_attack_by_threat_model[
            attack.threat_model
        ] = attack

group_result_rows = []

for attack in best_attack_by_threat_model.values():
    for group_column in [
        "binary_group",
        "family_group",
    ]:
        for group_value, group_data in target_mia_data.groupby(
            group_column
        ):
            membership_counts = (
                group_data["membership"].value_counts()
            )

            if (
                membership_counts.get(0, 0) < 30
                or membership_counts.get(1, 0) < 30
            ):
                continue

            group_result_rows.append(
                evaluate_calibrated_attack(
                    attack,
                    group_data,
                    subset_name=(
                        f"{group_column}={group_value}"
                    ),
                    include_confidence_intervals=True,
                )
            )

mia_by_group = pd.DataFrame(group_result_rows)

mia_by_group.to_csv(
    RESULTS_DIR / "mia_by_group.csv",
    index=False,
)

display(mia_by_group)


## 10. Save diagnostic distributions and experiment manifests

In [ ]:
# Save member and non-member distribution data
distribution_data = target_mia_data[[
    "source",
    "membership",
    "binary_group",
    "family_group",
    "prob_attack",
    "confidence",
    "loss",
    "correctness",
]].copy()

distribution_data.to_csv(
    RESULTS_DIR / "member_nonmember_distribution.csv",
    index=False,
)

In [ ]:
# Plot confidence and loss distributions
def plot_membership_distribution(
    column: str,
    title: str,
) -> None:
    plt.figure(figsize=(7, 4))

    plt.hist(
        target_mia_data.loc[
            target_mia_data["membership"] == 0,
            column,
        ],
        bins=50,
        alpha=0.55,
        density=True,
        label="Non-member",
    )
    plt.hist(
        target_mia_data.loc[
            target_mia_data["membership"] == 1,
            column,
        ],
        bins=50,
        alpha=0.55,
        density=True,
        label="Member",
    )

    plt.xlabel(column)
    plt.ylabel("Density")
    plt.title(title)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR
        / f"{column}_member_vs_nonmember.png",
        dpi=160,
    )
    plt.show()

plot_membership_distribution(
    "confidence",
    "Target MLP confidence by membership",
)
plot_membership_distribution(
    "loss",
    "Target MLP loss by membership",
)

In [ ]:
# Save reproducibility configuration
configuration = {
    "protocol_version": "ROADMAP_REVISED_V2",
    "seed": SEED,
    "run_mode": RUN_MODE,
    "number_of_shadow_models": len(SHADOW_SEEDS),
    "shadow_seeds": SHADOW_SEEDS,
    "calibration_shadow_seed": calibration_shadow_seed,
    "bootstrap_n": BOOTSTRAP_N,
    "split": {
        "target_train": 0.70,
        "target_validation": 0.10,
        "shadow_pool": 0.20,
    },
    "target_model_parameters": (
        target_model.get_params()
    ),
    "target_training_diagnostics": (
        target_training_diagnostics.iloc[0].to_dict()
    ),
    "shadow_training_diagnostics": (
        shadow_run_summary.to_dict(orient="records")
    ),
}

with (
    RESULTS_DIR / "config.json"
).open("w") as file:
    json.dump(
        configuration,
        file,
        indent=2,
        default=str,
    )


In [ ]:
# Save the final experiment manifest
experiment_manifest = {
    "experiment": (
        "02_03_mia_ready_baseline_and_audit"
    ),
    "protocol_version": "ROADMAP_REVISED_V2",
    "created_at_unix": time.time(),
    "run_mode": RUN_MODE,
    "number_of_shadow_models": len(SHADOW_SEEDS),
    "shadow_seeds": SHADOW_SEEDS,
    "calibration_shadow_seed": calibration_shadow_seed,
    "bootstrap_n": BOOTSTRAP_N,
    "dataset": {
        "train_file": str(TRAIN_FILE),
        "test_file": str(TEST_FILE),
        "train_sha256": calculate_sha256(
            TRAIN_FILE
        ),
        "test_sha256": calculate_sha256(
            TEST_FILE
        ),
    },
    "partitions": split_summary.to_dict(
        orient="records"
    ),
    "selected_ids_threshold": best_threshold,
    "target_training_diagnostics": (
        target_training_diagnostics.iloc[0].to_dict()
    ),
    "shadow_training_diagnostics": (
        shadow_run_summary.to_dict(orient="records")
    ),
    "target_model_path": str(
        MODEL_DIR / "target_mlp.joblib"
    ),
    "target_preprocessor_path": str(
        PREPROCESSOR_DIR
        / "target_mlp_preprocessor.joblib"
    ),
    "mia_calibration_rule": (
        "The last shadow model was reserved entirely for "
        "MIA threshold calibration. All other shadow models "
        "were used for attacker training."
    ),
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}

with (
    MANIFEST_DIR
    / "baseline_mia_combined_manifest.json"
).open("w") as file:
    json.dump(
        experiment_manifest,
        file,
        indent=2,
        default=str,
    )


In [ ]:
# Conservative result interpretation
best_result = (
    mia_summary
    .sort_values(
        "mia_auc",
        ascending=False,
    )
    .iloc[0]
)

best_auc = float(best_result["mia_auc"])

if best_auc < 0.55:
    interpretation = (
        "Measurable membership leakage is weak under "
        "the specified shadow-calibrated MIA protocol."
    )
elif best_auc < 0.65:
    interpretation = (
        "The membership signal is above chance but modest. "
        "Later claims must remain conservative."
    )
else:
    interpretation = (
        "The membership signal is meaningfully above chance "
        "and can support a later DP-SGD leakage comparison."
    )

print("Best target MIA AUC:", round(best_auc, 4))
print(interpretation)
print("Outputs saved to:", RESULTS_DIR)

if RUN_MODE == "smoke":
    print(
        "\nNext: change RUN_MODE to 'final', restart the "
        "runtime, and run every cell again."
    )